In [87]:
import polars
import polars as pl
import numpy as np
DATA_BASE_PATH = "data/"

In [88]:
data = polars.read_csv(DATA_BASE_PATH + "node_information.csv")

In [89]:
# Read txt file containing the edges 
edges_df = polars.read_csv(DATA_BASE_PATH + "train.txt", separator=" ", has_header=False, new_columns=["a", "b", "label"])
edges_df.shape

(10496, 3)

### Feature source

In [90]:
# Create node_feature_mapping from node_id to feature vector
node_feature_mapping = {}
for row in data.iter_rows():
    node_id = row[0]
    features = np.array(row[1:])
    node_feature_mapping[node_id] = features

Standard interface to create dataset

In [91]:
def build_dataset(edges, labels, feature_builder):
    X = []
    Y = []
    unseen_nodes_count = 0
    for i in range(edges.shape[0]):
        a, b = edges[i]
        edge_features = feature_builder(a, b)
        if edge_features is not None:
            X.append(edge_features)
            Y.append(labels[i])
        else:
            unseen_nodes_count += 1
    print(f"Rows with >=1 unseen node: {unseen_nodes_count} / {edges.shape[0]}")
    return np.array(X), np.array(Y)

### Feature engineering

In [92]:
def raw_feature_builder(node_a, node_b):
    features_a = node_feature_mapping.get(node_a)
    features_b = node_feature_mapping.get(node_b)
    if features_a is None or features_b is None:
        return None 
    return np.concatenate([features_a, features_b])

In [93]:
data = edges_df.to_numpy()
edges = data[:, :2]  # columns: a, b
labels = data[:, 2]  # column: label

X, Y = build_dataset(edges, labels, raw_feature_builder)

# X, Y = raw_diff_dataset()
print(X.shape, Y.shape)
# Split test and train
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, random_state=42)

Rows with >=1 unseen node: 7 / 10496
(10489, 1864) (10489,)


In [94]:
from sklearn.metrics import classification_report

def evaluate_model(model, x, y,) :
    y_predicted = model.predict(x)
    report = classification_report(y, y_predicted)
    print(report)
    

### SVM baseline

In [95]:
from sklearn.svm import SVC

clf = SVC()
clf.fit(X_train, Y_train)

SVC()

In [96]:
evaluate_model(clf, X_test, Y_test)

              precision    recall  f1-score   support

           0       0.60      0.73      0.66      1554
           1       0.67      0.52      0.59      1593

    accuracy                           0.63      3147
   macro avg       0.63      0.63      0.62      3147
weighted avg       0.63      0.63      0.62      3147



### XGboost baseline

In [97]:
from xgboost import XGBClassifier

clf = XGBClassifier()
clf.fit(X_train, Y_train)



XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, random_state=None, ...)

In [98]:
evaluate_model(clf, X_test, Y_test)

              precision    recall  f1-score   support

           0       0.58      0.77      0.66      1554
           1       0.67      0.46      0.54      1593

    accuracy                           0.61      3147
   macro avg       0.63      0.61      0.60      3147
weighted avg       0.63      0.61      0.60      3147



### Logisitic regression

In [99]:
## Logistic Regression
from sklearn.linear_model import LogisticRegression
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, Y_train)
evaluate_model(clf, X_test, Y_test)

              precision    recall  f1-score   support

           0       0.60      0.64      0.62      1554
           1       0.62      0.58      0.60      1593

    accuracy                           0.61      3147
   macro avg       0.61      0.61      0.61      3147
weighted avg       0.61      0.61      0.61      3147



In [110]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from karateclub import DeepWalk
import networkx as nx

edges_np = edges_df.to_numpy()
edges_train_np, edges_test_np = train_test_split(
    edges_np,
    test_size=0.1,
    random_state=42,
    shuffle=True,
    stratify=edges_np[:, 2],
)

edges_train = pl.DataFrame(edges_train_np, schema=["a", "b", "label"]).with_columns(
    pl.col("a").cast(pl.Int64),
    pl.col("b").cast(pl.Int64),
    pl.col("label").cast(pl.Int64),
)
edges_test = pl.DataFrame(edges_test_np, schema=["a", "b", "label"]).with_columns(
    pl.col("a").cast(pl.Int64),
    pl.col("b").cast(pl.Int64),
    pl.col("label").cast(pl.Int64),
)

print("train/test sizes:", edges_train.shape, edges_test.shape)

train_pos = edges_train.filter(pl.col("label") == 1)
unique_nodes = set(train_pos["a"].to_list()) | set(train_pos["b"].to_list())
node_mapping = {node_id: i for i, node_id in enumerate(sorted(unique_nodes))}

G = nx.Graph()
for row in train_pos.iter_rows(named=True):
    u = node_mapping[row["a"]]
    v = node_mapping[row["b"]]
    G.add_edge(u, v)

print(f"Graph nodes (train positives): {G.number_of_nodes()}")
print(f"Graph edges (train positives): {G.number_of_edges()}")

model = DeepWalk()
model.fit(G)
embeddings = model.get_embedding()
emb_dim = embeddings.shape[1]
print("Embeddings shape:", embeddings.shape)

def get_node_embedding(node_id):
    mapped_id = node_mapping.get(node_id)
    if mapped_id is None:
        return np.ones(emb_dim, dtype=np.float32)
    return embeddings[mapped_id]

def embedding_feature_builder(node_a, node_b):
    emb_a = get_node_embedding(node_a)
    emb_b = get_node_embedding(node_b)
    return np.concatenate([emb_a, emb_b])

def hybrid_feature_builder(node_a, node_b):
    raw_a = node_feature_mapping.get(node_a)
    raw_b = node_feature_mapping.get(node_b)
    if raw_a is None or raw_b is None:
        return None 
    emb_a = get_node_embedding(node_a)
    emb_b = get_node_embedding(node_b)
    emb = np.array([np.dot(emb_a, emb_b)])
    return np.concatenate([raw_a, raw_b, emb_a, emb_b])



X_train_emb, Y_train_emb = build_dataset(edges_train.to_numpy()[:, :2], edges_train.to_numpy()[:, 2], hybrid_feature_builder)
X_test_emb, Y_test_emb = build_dataset(edges_test.to_numpy()[:, :2], edges_test.to_numpy()[:, 2], hybrid_feature_builder)
print("X_train/X_test:", X_train_emb.shape, X_test_emb.shape)


train/test sizes: (9446, 3) (1050, 3)
Graph nodes (train positives): 3428
Graph edges (train positives): 4723
Embeddings shape: (3428, 128)
Rows with >=1 unseen node: 6 / 9446
Rows with >=1 unseen node: 1 / 1050
X_train/X_test: (9440, 2120) (1049, 2120)


In [111]:
clf = XGBClassifier()
clf.fit(X_train_emb, Y_train_emb)
evaluate_model(clf, X_test_emb, Y_test_emb)
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_emb, Y_train_emb)
evaluate_model(clf, X_test_emb, Y_test_emb)

              precision    recall  f1-score   support

           0       0.56      0.93      0.70       524
           1       0.79      0.27      0.40       525

    accuracy                           0.60      1049
   macro avg       0.67      0.60      0.55      1049
weighted avg       0.67      0.60      0.55      1049

              precision    recall  f1-score   support

           0       0.58      0.66      0.62       524
           1       0.60      0.52      0.56       525

    accuracy                           0.59      1049
   macro avg       0.59      0.59      0.59      1049
weighted avg       0.59      0.59      0.59      1049

